# SHV R62 | 公开基准稳定性与反向峰值归因

R61-FIX1 已完成 FULL/TILED/LIGER，同次测试下 TILED 与 Liger 的 S512 显存相差仅约 0.119 MiB。R62 **不是**新压缩方法，不声称 SHV 独创优势。

- 同一 GPT-2 Small / B=4 / S=512 / AdamW8bit / Checkpointing 的三次独立进程配对重复
- 各 3 步预热、7 步有效计时，且交替运行顺序；额外的 Profiler 运行不参与速度判定
- 固定来源与协议哈希；自动导出结果 ZIP 与 SHA-256
- 不重新运行 R1–R61，也不触碰正式 Guard 或长训练


In [ ]:
import importlib.util, subprocess, sys, torch
for package, module in [('transformers','transformers'),('bitsandbytes','bitsandbytes'),('liger-kernel','liger_kernel')]:
    if importlib.util.find_spec(module) is None:
        res = subprocess.run([sys.executable,'-m','pip','install','-q',package],check=False)
        if res.returncode:
            print('DEPENDENCY_WARN',package,res.returncode)
print('CUDA=',torch.cuda.is_available(), 'GPU=',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')


In [ ]:
from pathlib import Path
import base64,hashlib,io,json,os,sys,zipfile
SOURCE_ZIP_SHA256 = '94840c534028a96691f353b9b5aed8647d27a062db091d4af56dd639df2b8e95'
SOURCE_ZIP_B64 = '...'
archive=base64.b64decode(SOURCE_ZIP_B64)
assert hashlib.sha256(archive).hexdigest()==SOURCE_ZIP_SHA256, 'Embedded source SHA mismatch'
ROOT=Path('/content/shv_r62_q33' if Path('/content').is_dir() else './shv_r62_q33').resolve()
ROOT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(archive)) as z:
    assert z.testzip() is None, 'Source ZIP CRC error'
    manifest=json.loads(z.read('PACKAGE_MANIFEST.json'))
    for f,sha in manifest['files'].items():
        assert hashlib.sha256(z.read(f)).hexdigest()==sha, f'Embedded mismatch: {f}'
    z.extractall(ROOT)
print('Verified R62 source files:',len(manifest['files']),'root:',ROOT)


In [ ]:
import os,subprocess,sys,json
ENV=os.environ.copy();ENV['PYTHONPATH']=str(ROOT)+os.pathsep+ENV.get('PYTHONPATH','')
t=subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],cwd=ROOT,env=ENV,check=False)
assert t.returncode==0, 'CPU mathematical/protocol contract failed'
RESULT_DIR=ROOT/'result_r62'
r=subprocess.run([sys.executable,'-m','shv_r62.driver',str(RESULT_DIR)],cwd=ROOT,env=ENV,check=False)
assert r.returncode==0, 'Driver failed before producing auditable results'
report=json.loads((RESULT_DIR/'R62_RESULTS'/'summary.json').read_text('utf-8'))
print('R62 STATUS=',report['status'])
print(json.dumps(report['aggregate'],ensure_ascii=False,indent=2))
RESULT_ZIP=RESULT_DIR/'SHV_R62_PROFILE_Q33_A0_RESULT_UPLOAD_TO_CHATGPT.zip'
RESULT_SHA=RESULT_DIR/'SHV_R62_PROFILE_Q33_A0_RESULT_UPLOAD_TO_CHATGPT.zip.sha256'
assert RESULT_ZIP.is_file() and RESULT_SHA.is_file()


In [ ]:
# Download both artifacts after execution (PASS / HOLD / NO_CUDA / ERROR always archived)
try:
    from google.colab import files
    files.download(str(RESULT_ZIP))
    files.download(str(RESULT_SHA))
except ImportError:
    print('Files:',RESULT_ZIP,RESULT_SHA)
